[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/paper_implementation/blob/main/mean_flow_ablation.ipynb)

# MeanFlow MNIST ablation

이전 실패 원인을 `conditioning`, `hyperparameter`, `optimizer`로 분리해서 비교한다.

실행 순서는 빠른 sanity check를 위해 `all_combined`부터 시작한다.

- 각 run 최대 1000 step
- 50 step마다 진단
- 100 step마다 샘플 저장
- 600 step에서 자동 조기 판정
- `interval_cosine < 0.60` 또는 `boundary_mse > 0.80`이면 해당 run을 종료하고 다음 ablation으로 넘어간다.


## 0. Setup / experiment table


In [ ]:
!pip -q install datasets tensorboard

import math
import os
import random
import time
from dataclasses import asdict, dataclass

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
from torch.func import jvp
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets as tv_datasets
from torchvision import transforms
from torchvision.transforms import ToTensor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Colab GPU runtime을 연결해 주세요.")

DEVICE = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.mha.set_fastpath_enabled(False)

MAX_STEPS = 1_000
EARLY_CHECK_STEP = 600

BATCH_SIZE = 128
DIAG_BATCH_SIZE = 64
LOG_EVERY = 50
DIAG_EVERY = 50
SAMPLE_EVERY = 100

EARLY_STOP_MIN_INTERVAL_COSINE = 0.60
EARLY_STOP_MAX_BOUNDARY_MSE = 0.80

P_MEAN = -0.4
P_STD = 1.0
DATA_PROPORTION = 0.75
NORM_P = 1.0
NUM_CLASSES = 10
SAMPLE_COUNT = 16

ROOT_DIR = "/content/meanflow_mnist_ablation"
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(ROOT_DIR, RUN_ID)
TB_DIR = os.path.join(RUN_DIR, "tensorboard")
SAMPLE_DIR = os.path.join(RUN_DIR, "samples")

os.makedirs(TB_DIR, exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)


@dataclass(frozen=True)
class Experiment:
    name: str
    conditional: bool
    optimizer: str
    adam_lr: float
    norm_eps: float
    muon_lr: float = 2e-2
    auxiliary_lr: float = 3e-4
    muon_momentum: float = 0.95


EXPERIMENTS = {
    "all_combined": Experiment(
        name="all_combined",
        conditional=True,
        optimizer="muon",
        adam_lr=1e-3,
        norm_eps=0.01,
    ),
    "baseline_original": Experiment(
        name="baseline_original",
        conditional=False,
        optimizer="adam",
        adam_lr=1e-3,
        norm_eps=1.0,
    ),
    "conditioning_only": Experiment(
        name="conditioning_only",
        conditional=True,
        optimizer="adam",
        adam_lr=1e-3,
        norm_eps=1.0,
    ),
    "hp_bundle_only": Experiment(
        name="hp_bundle_only",
        conditional=False,
        optimizer="adam",
        adam_lr=3e-4,
        norm_eps=0.01,
    ),
    "muon_only": Experiment(
        name="muon_only",
        conditional=False,
        optimizer="muon",
        adam_lr=1e-3,
        norm_eps=1.0,
    ),
    "lr_only": Experiment(
        name="lr_only",
        conditional=False,
        optimizer="adam",
        adam_lr=3e-4,
        norm_eps=1.0,
    ),
    "normeps_only": Experiment(
        name="normeps_only",
        conditional=False,
        optimizer="adam",
        adam_lr=1e-3,
        norm_eps=0.01,
    ),
}

EXPERIMENTS_TO_RUN = [
    "all_combined",
    "baseline_original",
    "conditioning_only",
    "hp_bundle_only",
    "muon_only",
]

print("RUN_DIR:", RUN_DIR)
print("Run order:", EXPERIMENTS_TO_RUN)


## 1. TensorBoard — 학습 전에 먼저 실행

이 셀은 학습 셀과 독립적이다. 아래 훈련을 시작하기 전에 먼저 실행한다.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/meanflow_mnist_ablation --reload_interval 5


## 2. MNIST


In [ ]:
to_tensor = ToTensor()


def transform_image(image):
    image = to_tensor(image)
    image = F.pad(
        image,
        (2, 2, 2, 2),
        value=0.0,
    )
    return image * 2.0 - 1.0


def hf_collate(batch):
    images = torch.stack(
        [transform_image(item["image"]) for item in batch]
    )
    labels = torch.tensor(
        [item["label"] for item in batch],
        dtype=torch.long,
    )
    return images, labels


try:
    dataset = load_dataset("ylecun/mnist")

    train_loader = DataLoader(
        dataset["train"],
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        num_workers=2,
        pin_memory=True,
        collate_fn=hf_collate,
    )
    test_loader = DataLoader(
        dataset["test"],
        batch_size=DIAG_BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        collate_fn=hf_collate,
    )
    print("MNIST: Hugging Face")

except Exception as error:
    print("HF failed:", repr(error))
    print("Fallback: torchvision")

    transform = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Pad(2),
            transforms.Normalize((0.5,), (0.5,)),
        ]
    )

    train_set = tv_datasets.MNIST(
        "/content/mnist_data",
        train=True,
        download=True,
        transform=transform,
    )
    test_set = tv_datasets.MNIST(
        "/content/mnist_data",
        train=False,
        download=True,
        transform=transform,
    )

    train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        num_workers=2,
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_set,
        batch_size=DIAG_BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )


## 3. Conditional / unconditional DiT


In [ ]:
class ScalarEmbedding(nn.Module):
    def __init__(self, hidden_dim, frequency_dim=128):
        super().__init__()
        self.frequency_dim = frequency_dim
        self.mlp = nn.Sequential(
            nn.Linear(frequency_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, value):
        half = self.frequency_dim // 2
        frequencies = torch.exp(
            -math.log(10_000.0)
            * torch.arange(
                half,
                device=value.device,
                dtype=value.dtype,
            )
            / half
        )
        angles = value[:, None] * frequencies[None, :] * 2.0 * math.pi
        embedding = torch.cat(
            [angles.cos(), angles.sin()],
            dim=-1,
        )
        return self.mlp(embedding)


class DiTBlock(nn.Module):
    def __init__(self, hidden_dim=224, heads=8):
        super().__init__()
        self.norm1 = nn.LayerNorm(
            hidden_dim,
            elementwise_affine=False,
            eps=1e-6,
        )
        self.norm2 = nn.LayerNorm(
            hidden_dim,
            elementwise_affine=False,
            eps=1e-6,
        )
        self.attention = nn.MultiheadAttention(
            hidden_dim,
            heads,
            batch_first=True,
        )
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * hidden_dim, hidden_dim),
        )
        self.modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_dim, 6 * hidden_dim),
        )
        nn.init.zeros_(self.modulation[-1].weight)
        nn.init.zeros_(self.modulation[-1].bias)

    def forward(self, tokens, condition):
        (
            shift1,
            scale1,
            gate1,
            shift2,
            scale2,
            gate2,
        ) = self.modulation(condition).chunk(6, dim=-1)

        attention_input = self.norm1(tokens)
        attention_input = (
            attention_input * (1.0 + scale1[:, None, :])
            + shift1[:, None, :]
        )
        attention_output, _ = self.attention(
            attention_input,
            attention_input,
            attention_input,
            need_weights=True,
        )
        tokens = tokens + gate1[:, None, :] * attention_output

        mlp_input = self.norm2(tokens)
        mlp_input = (
            mlp_input * (1.0 + scale2[:, None, :])
            + shift2[:, None, :]
        )
        return tokens + gate2[:, None, :] * self.mlp(mlp_input)


class MeanFlowDiT(nn.Module):
    def __init__(
        self,
        conditional,
        hidden_dim=224,
        depth=4,
        heads=8,
        patch_size=4,
    ):
        super().__init__()
        self.conditional = conditional
        self.patch_size = patch_size

        self.patch_embed = nn.Conv2d(
            1,
            hidden_dim,
            patch_size,
            patch_size,
        )
        self.position = nn.Parameter(
            torch.zeros(1, 64, hidden_dim)
        )
        self.time_embed = ScalarEmbedding(hidden_dim)
        self.interval_embed = ScalarEmbedding(hidden_dim)

        if conditional:
            self.class_embed = nn.Embedding(
                NUM_CLASSES,
                hidden_dim,
            )
        else:
            self.class_embed = None

        self.blocks = nn.ModuleList(
            [DiTBlock(hidden_dim, heads) for _ in range(depth)]
        )
        self.final_norm = nn.LayerNorm(
            hidden_dim,
            elementwise_affine=False,
            eps=1e-6,
        )
        self.final_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_dim, 2 * hidden_dim),
        )
        self.output = nn.Linear(
            hidden_dim,
            patch_size * patch_size,
        )

        nn.init.normal_(self.position, std=0.02)
        if self.class_embed is not None:
            nn.init.normal_(self.class_embed.weight, std=0.02)
        nn.init.zeros_(self.final_modulation[-1].weight)
        nn.init.zeros_(self.final_modulation[-1].bias)
        nn.init.zeros_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, image, t, h, labels=None):
        batch_size = image.shape[0]
        tokens = self.patch_embed(image)
        tokens = tokens.flatten(2).transpose(1, 2)
        tokens = tokens + self.position

        condition = (
            self.time_embed(t)
            + self.interval_embed(h)
        )

        if self.conditional:
            if labels is None:
                raise ValueError("Conditional model needs labels.")
            condition = condition + self.class_embed(labels)

        for block in self.blocks:
            tokens = block(tokens, condition)

        shift, scale = self.final_modulation(condition).chunk(2, dim=-1)
        tokens = self.final_norm(tokens)
        tokens = (
            tokens * (1.0 + scale[:, None, :])
            + shift[:, None, :]
        )

        patches = self.output(tokens)
        patches = patches.view(
            batch_size,
            8,
            8,
            self.patch_size,
            self.patch_size,
            1,
        )
        image = torch.einsum(
            "nhwpqc->nchpwq",
            patches,
        )
        return image.reshape(batch_size, 1, 32, 32)


## 4. MeanFlow objective / optimizer


In [ ]:
def logit_normal(batch_size):
    value = torch.randn(
        batch_size,
        device=DEVICE,
    )
    return torch.sigmoid(
        value * P_STD + P_MEAN
    )


def sample_training_tuple(images):
    batch_size = images.shape[0]
    time_a = logit_normal(batch_size)
    time_b = logit_normal(batch_size)
    t = torch.maximum(time_a, time_b)
    r = torch.minimum(time_a, time_b)

    boundary_count = int(
        batch_size * DATA_PROPORTION
    )
    r[:boundary_count] = t[:boundary_count]

    noise = torch.randn_like(images)
    t_image = t[:, None, None, None]
    z_t = (
        (1.0 - t_image) * images
        + t_image * noise
    )
    velocity = noise - images
    return z_t, velocity, r, t


def meanflow_outputs(
    model,
    z_t,
    velocity,
    r,
    t,
    labels,
):
    def u_fn(z_value, t_value, r_value):
        return model(
            z_value,
            t_value,
            t_value - r_value,
            labels,
        )

    prediction, derivative = jvp(
        u_fn,
        (z_t, t, r),
        (
            velocity,
            torch.ones_like(t),
            torch.zeros_like(r),
        ),
    )

    interval = (t - r)[:, None, None, None]
    target = (
        velocity
        - interval * derivative
    ).detach()

    return prediction, target


def meanflow_loss(
    prediction,
    target,
    norm_eps,
):
    error = (prediction - target).pow(2)
    per_sample_sse = error.flatten(1).sum(1)

    with torch.no_grad():
        weight = 1.0 / (
            per_sample_sse + norm_eps
        ).pow(NORM_P)

    adaptive_loss = (
        per_sample_sse * weight
    ).mean()
    raw_mse = error.mean()

    return adaptive_loss, raw_mse


@torch.no_grad()
def orthogonalize(matrix, steps=5, eps=1e-7):
    value = matrix.float()
    transpose = value.shape[0] > value.shape[1]

    if transpose:
        value = value.T

    value = value / (value.norm() + eps)

    a = 3.4445
    b = -4.775
    c = 2.0315

    for _ in range(steps):
        gram = value @ value.T
        value = (
            a * value
            + (b * gram + c * gram @ gram) @ value
        )

    if transpose:
        value = value.T

    return value.to(matrix.dtype)


class MuonFallback(torch.optim.Optimizer):
    def __init__(self, params, lr, momentum):
        defaults = {
            "lr": lr,
            "momentum": momentum,
        }
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for parameter in group["params"]:
                if parameter.grad is None:
                    continue

                state = self.state[parameter]
                buffer = state.setdefault(
                    "momentum_buffer",
                    torch.zeros_like(parameter.grad),
                )
                momentum = group["momentum"]
                buffer.mul_(momentum).add_(parameter.grad)

                update = parameter.grad + momentum * buffer
                update = orthogonalize(update)
                scale = math.sqrt(
                    max(
                        1.0,
                        parameter.shape[0] / parameter.shape[1],
                    )
                )
                parameter.add_(
                    update * scale,
                    alpha=-group["lr"],
                )


def build_optimizers(model, experiment):
    if experiment.optimizer == "adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=experiment.adam_lr,
            betas=(0.9, 0.99),
            eps=1e-8,
        )
        return [optimizer], "Adam"

    muon_params = []
    auxiliary_params = []

    for name, parameter in model.named_parameters():
        use_muon = (
            parameter.ndim == 2
            and (
                ".attention." in name
                or ".mlp." in name
            )
        )

        if use_muon:
            muon_params.append(parameter)
        else:
            auxiliary_params.append(parameter)

    if hasattr(torch.optim, "Muon"):
        muon = torch.optim.Muon(
            muon_params,
            lr=experiment.muon_lr,
            momentum=experiment.muon_momentum,
            weight_decay=0.0,
        )
        backend = "torch.optim.Muon"
    else:
        muon = MuonFallback(
            muon_params,
            lr=experiment.muon_lr,
            momentum=experiment.muon_momentum,
        )
        backend = "MuonFallback"

    adamw = torch.optim.AdamW(
        auxiliary_params,
        lr=experiment.auxiliary_lr,
        betas=(0.9, 0.99),
        eps=1e-8,
        weight_decay=0.0,
    )

    return [muon, adamw], backend + " + AdamW"


## 5. Fixed diagnostics / sample inputs


In [ ]:
diag_images, diag_labels = next(iter(test_loader))
diag_images = diag_images.to(DEVICE)
diag_labels = diag_labels.to(DEVICE)

fixed_generator = torch.Generator(device=DEVICE)
fixed_generator.manual_seed(SEED + 1000)

diag_noise = torch.randn(
    diag_images.shape,
    generator=fixed_generator,
    device=DEVICE,
)

diag_a = torch.sigmoid(
    torch.randn(
        DIAG_BATCH_SIZE,
        generator=fixed_generator,
        device=DEVICE,
    )
    * P_STD
    + P_MEAN
)
diag_b = torch.sigmoid(
    torch.randn(
        DIAG_BATCH_SIZE,
        generator=fixed_generator,
        device=DEVICE,
    )
    * P_STD
    + P_MEAN
)

diag_t = torch.maximum(diag_a, diag_b)
diag_r = torch.minimum(diag_a, diag_b)

boundary_count = int(
    DIAG_BATCH_SIZE * DATA_PROPORTION
)
diag_r[:boundary_count] = diag_t[:boundary_count]

fixed_noise = torch.randn(
    SAMPLE_COUNT,
    1,
    32,
    32,
    generator=fixed_generator,
    device=DEVICE,
)

label_rng = random.Random(SEED + 2000)
fixed_labels_list = [
    label_rng.randrange(NUM_CLASSES)
    for _ in range(SAMPLE_COUNT)
]
fixed_labels = torch.tensor(
    fixed_labels_list,
    device=DEVICE,
)


## 6. Diagnostics / sampling / early-stop rule


In [ ]:
def grad_norm(model):
    total = 0.0

    for parameter in model.parameters():
        if parameter.grad is not None:
            total += (
                parameter.grad
                .detach()
                .float()
                .pow(2)
                .sum()
                .item()
            )

    return math.sqrt(total)


@torch.no_grad()
def diagnostics(model, experiment):
    model.eval()

    t_image = diag_t[:, None, None, None]
    z_t = (
        (1.0 - t_image) * diag_images
        + t_image * diag_noise
    )
    velocity = diag_noise - diag_images

    labels = (
        diag_labels
        if experiment.conditional
        else None
    )

    prediction, target = meanflow_outputs(
        model,
        z_t,
        velocity,
        diag_r,
        diag_t,
        labels,
    )

    mse = (
        (prediction - target)
        .pow(2)
        .flatten(1)
        .mean(1)
    )
    cosine = F.cosine_similarity(
        prediction.flatten(1),
        target.flatten(1),
        dim=1,
    )

    interval_mask = diag_r < diag_t
    boundary_mask = diag_r == diag_t

    result = {
        "mse": mse.mean().item(),
        "interval_cosine": cosine[interval_mask].mean().item(),
        "boundary_mse": mse[boundary_mask].mean().item(),
    }

    model.train()
    return result


@torch.no_grad()
def save_samples(model, experiment, step):
    model.eval()

    labels = (
        fixed_labels
        if experiment.conditional
        else None
    )
    ones = torch.ones(
        SAMPLE_COUNT,
        device=DEVICE,
    )

    average_velocity = model(
        fixed_noise,
        ones,
        ones,
        labels,
    )
    generated = (
        fixed_noise - average_velocity
    ).clamp(-1.0, 1.0)
    generated = (generated + 1.0) / 2.0

    run_sample_dir = os.path.join(
        SAMPLE_DIR,
        experiment.name,
    )
    os.makedirs(run_sample_dir, exist_ok=True)

    figure, axes = plt.subplots(
        4,
        4,
        figsize=(7, 7),
    )

    for index, axis in enumerate(axes.flat):
        axis.imshow(
            generated[index, 0].cpu(),
            cmap="gray",
            vmin=0.0,
            vmax=1.0,
        )

        if experiment.conditional:
            axis.set_title(
                f"y={fixed_labels_list[index]}",
                fontsize=9,
            )
        else:
            axis.set_title(
                "unconditional",
                fontsize=8,
            )

        axis.axis("off")

    figure.suptitle(
        f"{experiment.name} — step {step}"
    )
    figure.tight_layout()
    figure.savefig(
        os.path.join(
            run_sample_dir,
            f"step_{step:05d}.png",
        ),
        dpi=150,
        bbox_inches="tight",
    )

    plt.close(figure)
    model.train()


def should_early_stop(metrics):
    bad_cosine = (
        metrics["interval_cosine"]
        < EARLY_STOP_MIN_INTERVAL_COSINE
    )
    bad_boundary = (
        metrics["boundary_mse"]
        > EARLY_STOP_MAX_BOUNDARY_MSE
    )
    return bad_cosine or bad_boundary


## 7. Run one ablation

각 run은 최대 1000 step이다. 600 step 진단에서 실패 기준을 만족하면 즉시 다음 run으로 넘어간다.


In [ ]:
def run_experiment(experiment):
    print()
    print("=" * 72)
    print(asdict(experiment))
    print("=" * 72)

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    model = MeanFlowDiT(
        conditional=experiment.conditional
    ).to(DEVICE)

    optimizers, backend = build_optimizers(
        model,
        experiment,
    )
    print("Optimizer:", backend)

    writer = SummaryWriter(
        os.path.join(
            TB_DIR,
            experiment.name,
        )
    )
    writer.add_text(
        "run/config",
        str(asdict(experiment)),
        0,
    )

    train_iterator = iter(train_loader)
    start_time = time.time()
    last_metrics = None
    stopped_early = False
    final_step = 0

    save_samples(
        model,
        experiment,
        0,
    )

    for step in range(1, MAX_STEPS + 1):
        final_step = step

        try:
            images, labels = next(train_iterator)
        except StopIteration:
            train_iterator = iter(train_loader)
            images, labels = next(train_iterator)

        images = images.to(
            DEVICE,
            non_blocking=True,
        )
        labels = labels.to(
            DEVICE,
            non_blocking=True,
        )

        training_labels = (
            labels
            if experiment.conditional
            else None
        )

        z_t, velocity, r, t = sample_training_tuple(images)
        prediction, target = meanflow_outputs(
            model,
            z_t,
            velocity,
            r,
            t,
            training_labels,
        )
        loss, raw_mse = meanflow_loss(
            prediction,
            target,
            experiment.norm_eps,
        )

        for optimizer in optimizers:
            optimizer.zero_grad(set_to_none=True)

        loss.backward()
        gradient_norm = grad_norm(model)

        for optimizer in optimizers:
            optimizer.step()

        if step % LOG_EVERY == 0:
            writer.add_scalar(
                "train/loss_adaptive",
                loss.item(),
                step,
            )
            writer.add_scalar(
                "train/raw_mse",
                raw_mse.item(),
                step,
            )
            writer.add_scalar(
                "train/grad_norm",
                gradient_norm,
                step,
            )
            writer.add_scalar(
                "run/elapsed_minutes",
                (time.time() - start_time) / 60.0,
                step,
            )

        if step % DIAG_EVERY == 0:
            last_metrics = diagnostics(
                model,
                experiment,
            )
            writer.add_scalar(
                "diagnostic/raw_mse",
                last_metrics["mse"],
                step,
            )
            writer.add_scalar(
                "diagnostic/interval_cosine",
                last_metrics["interval_cosine"],
                step,
            )
            writer.add_scalar(
                "diagnostic/boundary_mse",
                last_metrics["boundary_mse"],
                step,
            )

            print(
                f"{experiment.name:20s} "
                f"step={step:04d} "
                f"loss={loss.item():.6f} "
                f"grad={gradient_norm:.4f} "
                f"mse={last_metrics['mse']:.4f} "
                f"cos={last_metrics['interval_cosine']:.4f} "
                f"boundary={last_metrics['boundary_mse']:.4f}"
            )

        if step % SAMPLE_EVERY == 0:
            save_samples(
                model,
                experiment,
                step,
            )

        if step == EARLY_CHECK_STEP:
            if last_metrics is None:
                last_metrics = diagnostics(
                    model,
                    experiment,
                )

            if should_early_stop(last_metrics):
                stopped_early = True
                print()
                print(
                    f"[EARLY STOP] {experiment.name} at step {step}"
                )
                print(
                    f"interval_cosine={last_metrics['interval_cosine']:.4f}, "
                    f"boundary_mse={last_metrics['boundary_mse']:.4f}"
                )
                print("다음 experiment으로 넘어갑니다.")
                break

        if step % 100 == 0:
            writer.flush()

    writer.add_text(
        "run/result",
        (
            f"final_step={final_step}, "
            f"stopped_early={stopped_early}"
        ),
        final_step,
    )
    writer.flush()
    writer.close()

    result = {
        "name": experiment.name,
        "final_step": final_step,
        "stopped_early": stopped_early,
    }

    if last_metrics is not None:
        result.update(last_metrics)

    del model
    del optimizers
    torch.cuda.empty_cache()

    return result


## 8. Run the ablation suite

`all_combined`가 가장 먼저 실행된다. 600 step에서 조기 종료된 run은 자동으로 다음 run으로 넘어간다.


In [ ]:
results = []

for experiment_name in EXPERIMENTS_TO_RUN:
    result = run_experiment(
        EXPERIMENTS[experiment_name]
    )
    results.append(result)

print()
print("Finished:", RUN_DIR)

for result in results:
    print(result)
